In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pywt  # pip install PyWavelets

## Task 1 - Computer Screen Detection using Hough Line Transform

In [ ]:
def make_lab():
    img = np.full((420, 640, 3), 200, np.uint8)
    for r in range(3):
        for c in range(4):
            if (r, c) == (1, 2):          # missing screen
                continue
            x, y = 40 + c * 150, 30 + r * 130
            cv2.rectangle(img, (x, y), (x + 120, y + 80), (40, 40, 40), -1)       # bezel
            on = (r + c) % 3 != 0                                                  # some screens are off
            cv2.rectangle(img, (x + 8, y + 8), (x + 112, y + 72), (230, 200, 120) if on else (15, 15, 15), -1)
            cv2.rectangle(img, (x + 50, y + 80), (x + 70, y + 100), (90, 90, 90), -1)  # stand
    return img

lab = cv2.imread("lab.jpg") if os.path.exists("lab.jpg") else make_lab()
gray = cv2.cvtColor(lab, cv2.COLOR_BGR2GRAY)
edges = cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 0), 20, 60)

lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=40, minLineLength=30, maxLineGap=5)
line_mask = np.zeros_like(gray)
vis = lab.copy()
for x1, y1, x2, y2 in lines[:, 0]:
    ang = abs(np.degrees(np.arctan2(y2 - y1, x2 - x1)))
    if ang < 5 or abs(ang - 90) < 5 or abs(ang - 180) < 5:       # screens have axis-aligned borders
        cv2.line(line_mask, (x1, y1), (x2, y2), 255, 2)
        cv2.line(vis, (x1, y1), (x2, y2), (0, 0, 255), 1)

# merge the fragmented Hough segments of each screen border into one blob -> one bounding box per screen
blobs = cv2.dilate(line_mask, np.ones((15, 15), np.uint8))
cnts, _ = cv2.findContours(blobs, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
boxes = [cv2.boundingRect(c) for c in cnts]
screens = [(x + 7, y + 7, w - 14, h - 14) for x, y, w, h in boxes if 1.0 < w / h < 2.0 and w * h > 3000]  # undo dilation padding
print("screens found:", len(screens))

In [ ]:
out = lab.copy()
for x, y, w, h in screens:
    on = gray[y:y + h, x:x + w].mean() > 80          # bright inside = screen is on
    cv2.rectangle(out, (x, y), (x + w, y + h), (0, 200, 0) if on else (0, 0, 255), 2)
    cv2.putText(out, "ON" if on else "OFF", (x + 5, y + 18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

EXPECTED = 12                                          # screens that should be in the lab (3 rows x 4)
print(f"found {len(screens)} of {EXPECTED} -> {EXPECTED - len(screens)} missing")

fig, ax = plt.subplots(1, 4, figsize=(20, 4))
for a, im, t in zip(ax, [cv2.cvtColor(lab, cv2.COLOR_BGR2RGB), edges, cv2.cvtColor(vis, cv2.COLOR_BGR2RGB), cv2.cvtColor(out, cv2.COLOR_BGR2RGB)],
                    ["Input", "Canny edges", "Hough lines (axis-aligned)", "Screens: ON (green) / OFF (red)"]):
    a.imshow(im, cmap="gray"); a.set_title(t); a.axis("off")
plt.show()

**Result:** Canny + probabilistic Hough recovers the (fragmented) straight bezel edges; keeping only near-horizontal/vertical lines and merging each screen's segments into one blob gives a bounding box per screen. Mean brightness inside each region classifies ON/OFF, and comparing the found count with the expected count flags missing monitors. Limitation: it says *how many* are missing, not *which*.

## Task 2 - Asset Tracking using SIFT

In [ ]:
rng = np.random.default_rng(0)

def make_template():
    t = np.full((120, 300, 3), 230, np.uint8)
    for r in range(4):                       # keycaps with different letters => distinctive features
        for c in range(10):
            x, y = 8 + c * 29, 8 + r * 28
            cv2.rectangle(t, (x, y), (x + 24, y + 22), (60, 60, 60), 2)
            cv2.putText(t, chr(65 + (r * 10 + c) % 26), (x + 5, y + 17), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (20, 20, 20), 1)
    return t

template = cv2.imread("template.jpg") if os.path.exists("template.jpg") else make_template()

if os.path.exists("scene.jpg"):
    scene = cv2.imread("scene.jpg")
else:
    scene = np.full((500, 700, 3), 180, np.uint8)
    for _ in range(25):                       # clutter
        p = rng.integers(0, 600, 2)
        cv2.rectangle(scene, tuple(map(int, p)), tuple(map(int, p + rng.integers(20, 90, 2))), tuple(int(v) for v in rng.integers(40, 200, 3)), -1)
    M = cv2.getRotationMatrix2D((150, 60), 20, 1.4)
    M[:, 2] += (250, 200)
    warped = cv2.warpAffine(template, M, (700, 500), borderValue=(0, 0, 0))
    mask = cv2.warpAffine(np.full(template.shape[:2], 255, np.uint8), M, (700, 500))
    scene[mask > 0] = warped[mask > 0]
    scene = np.clip(scene + rng.normal(0, 6, scene.shape), 0, 255).astype(np.uint8)

sift = cv2.SIFT_create()
kp1, d1 = sift.detectAndCompute(cv2.cvtColor(template, cv2.COLOR_BGR2GRAY), None)
kp2, d2 = sift.detectAndCompute(cv2.cvtColor(scene, cv2.COLOR_BGR2GRAY), None)

# Lowe ratio test
good = [m for m, n in cv2.BFMatcher().knnMatch(d1, d2, k=2) if m.distance < 0.75 * n.distance]
print(f"keypoints: template={len(kp1)} scene={len(kp2)}, good matches={len(good)}")

print("ASSET RECOGNISED" if len(good) >= 10 else "not found")

match_img = cv2.drawMatches(template, kp1, scene, kp2, good, None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

fig, ax = plt.subplots(1, 1, figsize=(16, 6))
ax.imshow(cv2.cvtColor(match_img, cv2.COLOR_BGR2RGB)); ax.set_title("SIFT matches: template (left) -> scene (right)"); ax.axis("off")
plt.show()

**Result:** SIFT keypoints are scale- and rotation-invariant, so the template is found even though the scene copy is rotated by 20 degrees and enlarged 1.4x. The ratio test removes ambiguous matches, so many surviving matches means the asset is present. For several asset types (monitor, keyboard, ...), repeat per template and count the ones with enough inliers.

## Task 3 - Anomaly Detection in Sensor Data using Wavelet Transform

In [ ]:
rng = np.random.default_rng(1)
t = np.arange(2048)
signal = np.sin(2 * np.pi * t / 128) + 0.3 * np.sin(2 * np.pi * t / 32) + rng.normal(0, 0.05, t.size)  # healthy machine
truth = np.zeros(t.size, bool)
for i in (500, 1100):                          # sudden spikes (e.g. impacts)
    signal[i:i + 3] += 2.5; truth[i:i + 3] = True
signal[1500:1560] += 0.8 * np.sin(2 * np.pi * t[1500:1560] / 6); truth[1500:1560] = True   # vibration burst

wavelet, level = "db4", 4
coeffs = pywt.wavedec(signal, wavelet, level=level, mode="periodization")  # periodization avoids edge artefacts
details = coeffs[1:]                           # cD4 ... cD1 (cD1 = finest scale)

d1 = np.abs(details[-1])                       # finest-scale detail: ~0 for smooth signal, large at sudden changes
bad = np.where(d1 > d1.mean() + 3 * d1.std())[0]
flag = np.zeros(t.size, bool)
for k in bad:
    flag[2 * k: 2 * k + 2] = True                  # each level-1 coefficient covers 2 samples

print("anomalies flagged near samples:", sorted(set(int(i) // 50 * 50 for i in np.flatnonzero(flag))))   # injected at ~500, ~1100, ~1500

fig, ax = plt.subplots(2, 1, figsize=(14, 6))
ax[0].plot(t, signal, lw=0.8); ax[0].fill_between(t, signal.min(), signal.max(), where=flag, color="red", alpha=0.3, label="detected")
ax[0].set_title("Sensor signal with detected anomalies"); ax[0].legend()
ax[1].plot(np.abs(details[-1]), lw=0.8); ax[1].set_title("|cD1| finest-scale detail coefficients (spikes = anomalies)")
plt.tight_layout(); plt.show()

**Result:** Normal behaviour lives in the low-frequency approximation, while abrupt changes (spikes, vibration bursts) produce large detail coefficients at fine/mid scales. Flagging coefficients more than 3 standard deviations above the mean flags those regions and maps them back to time, without needing a model of the healthy signal. In real use, replace the synthetic signal with the sensor stream (windowed for real-time) and tune `wavelet`, `level` and the 3-sigma threshold.

## Task 4 - Object Recognition (Using Video) with SIFT

In [ ]:
VIDEO = None  # e.g. "video.mp4"

def make_frames():
    frames = []
    # (scale, angle, occluded?)
    for scale, angle, occ in [(1.0, 0, False), (1.6, 15, False), (0.7, -30, False), (1.3, 45, True), (1.2, 160, False)]:
        f = np.full((500, 700, 3), 170, np.uint8)
        M = cv2.getRotationMatrix2D((150, 60), angle, scale); M[:, 2] += (250, 200)
        warped = cv2.warpAffine(template, M, (700, 500))
        mask = cv2.warpAffine(np.full(template.shape[:2], 255, np.uint8), M, (700, 500))
        f[mask > 0] = warped[mask > 0]
        if occ:
            cv2.rectangle(f, (300, 150), (420, 330), (60, 90, 40), -1)   # something blocks part of the object
        frames.append(f)
    return frames

if VIDEO:
    cap, frames = cv2.VideoCapture(VIDEO), []
    while len(frames) < 30 and cap.isOpened():
        ok, fr = cap.read()
        if not ok: break
        frames.append(fr)
else:
    frames = make_frames()

kp_t, d_t = sift.detectAndCompute(cv2.cvtColor(template, cv2.COLOR_BGR2GRAY), None)
th, tw = template.shape[:2]

def recognise(frame):
    kp, d = sift.detectAndCompute(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), None)
    good = [m for m, n in cv2.BFMatcher().knnMatch(d_t, d, k=2) if m.distance < 0.75 * n.distance]
    if len(good) < 10:
        return frame, 0
    src = np.float32([kp_t[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst = np.float32([kp[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    H, inl = cv2.findHomography(src, dst, cv2.RANSAC, 5.0)
    if H is None or inl.sum() < 10:
        return frame, 0
    box = cv2.perspectiveTransform(np.float32([[0, 0], [tw, 0], [tw, th], [0, th]]).reshape(-1, 1, 2), H)
    out = frame.copy()
    cv2.polylines(out, [np.int32(box)], True, (0, 255, 0), 3)        # bounding box around the object
    return out, int(inl.sum())

results = [recognise(f) for f in frames]
fig, ax = plt.subplots(1, len(results), figsize=(4 * len(results), 3))
for a, (img, n), i in zip(np.atleast_1d(ax), results, range(len(results))):
    a.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); a.set_title(f"Frame {i}: {n} inliers"); a.axis("off")
plt.show()

**Result:** SIFT is scale- and rotation-invariant, so the keyboard is boxed at 0.7x-1.6x size and at any angle. With partial occlusion the visible part still gives enough matching keypoints, so it is still found (fewer inliers). Frames with fewer than 10 inliers are treated as "object not present".

## Task 5 - Panoramic Image with SIFT

In [ ]:
photo = cv2.imread("../05/dog.jpeg")
h, w = photo.shape[:2]
left, right = photo[:, : int(0.6 * w)], photo[:, int(0.4 * w):]       # overlapping views

def stitch(a, b):
    # warp b onto a using SIFT matches
    ka, da = sift.detectAndCompute(cv2.cvtColor(a, cv2.COLOR_BGR2GRAY), None)
    kb, db = sift.detectAndCompute(cv2.cvtColor(b, cv2.COLOR_BGR2GRAY), None)
    good = [m for m, n in cv2.BFMatcher().knnMatch(db, da, k=2) if m.distance < 0.75 * n.distance]
    src = np.float32([kb[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst = np.float32([ka[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    H, _ = cv2.findHomography(src, dst, cv2.RANSAC, 5.0)
    pano = cv2.warpPerspective(b, H, (a.shape[1] + b.shape[1], max(a.shape[0], b.shape[0])))
    pano[: a.shape[0], : a.shape[1]] = a
    print("matches used:", len(good))
    return pano

pano = stitch(left, right)
cols = np.where(pano.sum(axis=(0, 2)) > 0)[0]                         # crop empty black area
pano = pano[:, : cols.max() + 1]

fig, ax = plt.subplots(1, 3, figsize=(18, 4), gridspec_kw={"width_ratios": [1, 1, 1.6]})
for a, im, t in zip(ax, [left, right, pano], ["Image 1", "Image 2", "Panorama"]):
    a.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); a.set_title(t); a.axis("off")
plt.show()

**Result:** SIFT keypoints in the overlapping strip are matched (ratio test), RANSAC estimates the homography that maps image 2 into image 1's frame, and `warpPerspective` places it so the two halves join into one wider image. Real panning photos also need exposure blending at the seam (e.g. `cv2.Stitcher`).

## Task 6 - Lane Detection using the Hough Line Transform

In [ ]:
def make_road():
    r = np.full((400, 640, 3), (90, 90, 90), np.uint8)
    cv2.rectangle(r, (0, 0), (640, 170), (200, 170, 120), -1)                                      # sky
    cv2.fillPoly(r, [np.array([[0, 400], [250, 170], [390, 170], [640, 400]])], (60, 60, 60))      # road
    cv2.line(r, (140, 400), (300, 170), (255, 255, 255), 6)                                        # left lane
    cv2.line(r, (520, 400), (340, 170), (255, 255, 255), 6)                                        # right lane
    return np.clip(r + np.random.default_rng(0).normal(0, 6, r.shape), 0, 255).astype(np.uint8)

road = cv2.imread("road.jpg") if os.path.exists("road.jpg") else make_road()
H_, W_ = road.shape[:2]

edges = cv2.Canny(cv2.GaussianBlur(cv2.cvtColor(road, cv2.COLOR_BGR2GRAY), (5, 5), 0), 50, 150)
roi = np.zeros_like(edges)
cv2.fillPoly(roi, [np.array([[0, H_], [W_ // 2 - 60, H_ // 2 + 20], [W_ // 2 + 60, H_ // 2 + 20], [W_, H_]])], 255)   # keep only the road area
edges_roi = cv2.bitwise_and(edges, roi)

lines = cv2.HoughLinesP(edges_roi, 1, np.pi / 180, 40, minLineLength=40, maxLineGap=20)
lane = road.copy()
for x1, y1, x2, y2 in lines[:, 0]:
    cv2.line(lane, (x1, y1), (x2, y2), (0, 0, 255), 4)

fig, ax = plt.subplots(1, 3, figsize=(18, 4))
for a, im, t in zip(ax, [cv2.cvtColor(road, cv2.COLOR_BGR2RGB), edges_roi, cv2.cvtColor(lane, cv2.COLOR_BGR2RGB)], ["Camera image", "Canny edges in road region", "Detected lane lines"]):
    a.imshow(im, cmap="gray"); a.set_title(t); a.axis("off")
plt.show()

**Result:** Canny finds edges, a trapezoid mask removes everything except the road ahead (sky, trees, buildings), and `HoughLinesP` returns the straight segments, which are the lane markings. Limitation: curved lanes and shadows need extra handling (slope filtering, curve fitting).

## Task 7 - Coins Detection and Counting using Hough Circles
Uses the coins photo from Lab 5.

In [ ]:
coins = cv2.imread("../05/water_coins.jpg")
gray_c = cv2.medianBlur(cv2.cvtColor(coins, cv2.COLOR_BGR2GRAY), 5)       # blur so the texture is not mistaken for edges

circles = cv2.HoughCircles(gray_c, cv2.HOUGH_GRADIENT, dp=1.2, minDist=35, param1=100, param2=25, minRadius=20, maxRadius=40)
res = coins.copy()
count = 0
if circles is not None:
    for x, y, r in np.uint16(np.around(circles[0])):
        cv2.circle(res, (x, y), r, (0, 255, 0), 3)
        cv2.circle(res, (x, y), 2, (0, 0, 255), 3)
        count += 1
cv2.putText(res, f"Coins: {count}", (10, 35), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
print("coins counted:", count)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].imshow(cv2.cvtColor(coins, cv2.COLOR_BGR2RGB)); ax[0].set_title("Input"); ax[0].axis("off")
ax[1].imshow(cv2.cvtColor(res, cv2.COLOR_BGR2RGB)); ax[1].set_title(f"Hough circles: {count} coins"); ax[1].axis("off")
plt.show()

**Result:** `HoughCircles` votes for circle centres from edge points, so coins of different sizes are found when the radius range covers them. `param2` (accumulator threshold) and `minDist` matter most: too low gives false circles, too high misses coins. Counting is just the number of circles returned.

## Task 8 - Smart Security System (zone intrusion)
A fixed security zone is watched in a generated "video": the empty scene is the background, and an object walks in. Moving object = difference from background; its boundary = contour; alarm if the boundary enters the zone. Set `VIDEO` to use a camera/video instead.

In [ ]:
ZONE = (330, 120, 500, 300)          # x1, y1, x2, y2 - restricted area

def make_security_frames():
    bg = np.full((360, 640, 3), 190, np.uint8)
    cv2.rectangle(bg, (0, 300), (640, 360), (140, 140, 140), -1)
    frames = []
    for x in (30, 130, 230, 330, 400):                    # object walks from left into the zone
        f = bg.copy()
        cv2.rectangle(f, (x, 190), (x + 70, 280), (30, 30, 200), -1)
        frames.append(f)
    return bg, frames

background, frames = make_security_frames()
bg_gray = cv2.GaussianBlur(cv2.cvtColor(background, cv2.COLOR_BGR2GRAY), (5, 5), 0)

def check(frame):
    g = cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    _, fg = cv2.threshold(cv2.absdiff(bg_gray, g), 30, 255, cv2.THRESH_BINARY)       # what changed
    fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)       # object boundaries
    out, alarm = frame.copy(), False
    for c in cnts:
        if cv2.contourArea(c) < 500: continue
        x, y, w, h = cv2.boundingRect(c)
        inside = x < ZONE[2] and x + w > ZONE[0] and y < ZONE[3] and y + h > ZONE[1]    # boxes overlap
        alarm |= inside
        cv2.drawContours(out, [c], -1, (0, 0, 255) if inside else (0, 200, 0), 2)
    cv2.rectangle(out, ZONE[:2], ZONE[2:], (255, 0, 0), 2)
    if alarm:
        cv2.putText(out, "ALARM!", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
    return out, alarm

res = [check(f) for f in frames]
fig, ax = plt.subplots(1, len(res), figsize=(4 * len(res), 3))
for a, (img, al), i in zip(ax, res, range(len(res))):
    a.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); a.set_title(f"Frame {i}: {'ALARM' if al else 'clear'}"); a.axis("off")
plt.show()

**Result:** Background subtraction + threshold isolates the moving object, `findContours` gives its boundary, and the alarm fires as soon as that boundary overlaps the predefined zone (blue box). For a live camera, run `check()` on each frame from `cv2.VideoCapture(0)`.